# Loss-Function Ablation under Leave-One-Patient-Out

Four losses, the **same** 89-fold LOPO protocol, the same architecture, the same seed,
the same splits. The only thing that changes is the loss.

| Arm | `pos_weight` | focal term $(1-p_t)^\gamma$ |
|---|---|---|
| `bce` | — | — |
| `weighted_bce` | $\min(n_0/n_1,\,12)$ | — |
| `focal` | — | $\gamma = 2$ |
| `weighted_focal` | $\min(n_0/n_1,\,12)$ | $\gamma = 2$ |

This is a **2 × 2 factorial design**: class weighting on/off crossed with focal
down-weighting on/off. That structure matters — it separates the effect of *each*
mechanism instead of comparing four arbitrary configurations, so you can say "weighting
helped, focal hurt" rather than just "arm 3 won".

### Why this experiment is worth 24 hours

Your single-split ablation found AUROC spanning only 0.0020 across these four losses
while severity κ_w spanned 0.20 — but that was one 15-patient test set and one seed, so
the honest reading was "suggestive". Repeating it under LOPO gives every arm 89 held-out
patients and lets you run **paired** significance tests, because every arm is evaluated
on exactly the same patients. That turns a suggestion into a result.

### Runtime, and how to survive it

4 arms × 89 folds = **356 training runs**, roughly 4 min each on a 4090 → **~24 hours**.
Three things make that manageable:

- Every fold is written to `{arm}/folds/{pid}.npz` and skipped on re-run.
- `ARMS_TO_RUN` lets you do one arm per session.
- `SEED_FOLDS_FROM` imports the `weighted_focal` folds you have already trained,
  saving 89 runs (~6 h) — **only valid if that run used this exact config**, which the
  notebook verifies before copying.

Set `MAX_FOLDS = 3` first and confirm all four arms train before committing the machine.

### One bug fixed relative to the LOPO notebook

A patient with `precision = 0` **and** `recall = 0` was being recorded as `NaN` F1 and
excluded from the mean. That is a genuine total failure worth 0.0, not an undefined
metric — `NaN` is only correct when the patient has no positive epochs at all. In your
last run this inflated macro F1 from 0.6118 to 0.6187. Fixed here, so numbers will differ
slightly from the earlier notebook.

## Cell 1 — Configuration

Everything writes under `ablation_results/`, one folder per arm plus a shared
`comparison/`:

```
ablation_results/
├── bce/            folds/ · figures/ · per_patient.csv · summary.csv
├── weighted_bce/   "
├── focal/          "
├── weighted_focal/ "
└── comparison/     csv/ · figures/     <- the cross-arm tables and plots
```

Config is byte-identical to the LOPO notebook apart from the arm list — same
`CHUNK_EPOCHS`, `VAL_K`, `EVAL_STRIDE`, `SEED`, `EPOCHS`. That is what makes the
comparison clean: any difference between arms is the loss, not the protocol.

In [ ]:
# =========================================================================
# CELL 1 — configuration
# =========================================================================
import os, re, gc, glob, json, time, random, shutil, warnings, sys, subprocess, importlib.util
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_auc_score, average_precision_score,
                             precision_recall_curve, matthews_corrcoef,
                             cohen_kappa_score, brier_score_loss, roc_curve)
from scipy.stats import pearsonr, spearmanr, wilcoxon

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

# ---- paths --------------------------------------------------------------
PROJECT_DIR = "/home/nishant/apneastudyproject"
CACHE_DIR   = os.environ.get("APNEA_CACHE",  os.path.join(PROJECT_DIR, "apnea_cache_resampled"))
OUT_DIR     = os.environ.get("APNEA_RESULT", os.path.join(PROJECT_DIR, "ablation_results"))
CMP_DIR     = os.path.join(OUT_DIR, "comparison")
CMP_CSV     = os.path.join(CMP_DIR, "csv")
CMP_FIG     = os.path.join(CMP_DIR, "figures")
for d in (OUT_DIR, CMP_DIR, CMP_CSV, CMP_FIG):
    os.makedirs(d, exist_ok=True)

# ---- the four arms (2x2 factorial: weighting x focal) -------------------
ARMS = ["bce", "weighted_bce", "focal", "weighted_focal"]
ARM_LABEL = {"bce": "BCE", "weighted_bce": "Weighted BCE",
             "focal": "Focal", "weighted_focal": "Weighted Focal"}
ARM_COLOR = {"bce": "#4c78a8", "weighted_bce": "#59a14f",
             "focal": "#e1913c", "weighted_focal": "#d1495b"}
BASELINE_ARM = "bce"            # paired tests compare every arm against this

ARMS_TO_RUN = ARMS              # e.g. ["bce"] to do one arm per session
MAX_FOLDS   = None              # 3 for a smoke run, None for all 89
RESUME      = True

# import folds already trained by apnea_lopo.ipynb (saves ~6 h). Config is verified.
SEED_FOLDS_FROM = {"weighted_focal": os.path.join(PROJECT_DIR,
                                                  "leave_one_out_output", "folds")}

def arm_dir(arm, *sub):
    return os.path.join(OUT_DIR, arm, *sub)

for a in ARMS:
    os.makedirs(arm_dir(a, "folds"), exist_ok=True)
    os.makedirs(arm_dir(a, "figures"), exist_ok=True)

# ---- protocol (identical to the LOPO notebook) --------------------------
SEED, VAL_K = 42, 12
WINDOW_SEC, CHUNK_EPOCHS = 30, 40
TRAIN_STRIDE, EVAL_STRIDE = 10, CHUNK_EPOCHS // 2

WIDTH, GRU_HIDDEN, GRU_LAYERS = 64, 96, 2
DILATIONS, KERNEL, SE_REDUCTION, DROPOUT = (1, 2, 4, 8, 16, 32), 9, 8, 0.2
STEM_STRIDE = 2
USE_BURDEN_HEAD, USE_BOUNDARY_HEAD = True, True

FOCAL_GAMMA, POS_WEIGHT_CAP = 2.0, 12.0
BURDEN_WEIGHT, BOUNDARY_WEIGHT, BOUNDARY_POS_WEIGHT = 0.3, 0.2, 8.0

EPOCHS, PATIENCE, BATCH_SIZE = 14, 5, 16
LR, WEIGHT_DECAY, GRAD_CLIP = 2e-3, 3e-4, 3.0
NUM_WORKERS = 0 if os.name == "nt" else 2

AUGMENT = True
NOISE_STD, SCALE_JITTER, CHANNEL_DROPOUT_P = 0.03, 0.10, 0.20
TIME_MASK_P, TIME_MASK_FRAC = 0.5, 0.10
FEAT_CLIP, NORM_SAMPLE_EPOCHS = 8.0, 8000

EPH_BINS = [5.0, 15.0, 30.0]
SEVERITY_NAMES = ["normal", "mild", "moderate", "severe"]
N_CALIB_BINS = 10

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"cache  {CACHE_DIR}")
print(f"output {OUT_DIR}")
print(f"device {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda"
                            else "  [!] CPU: ~20x slower, not viable for 356 folds"))
print(f"arms   {ARMS_TO_RUN}")

## Cell 2 — Load the pre-built cache

Verbatim from the LOPO notebook, so the cohort, the exclusions and the
events-per-hour definition are identical across the two studies.

In [ ]:
# =========================================================================
# CELL 2 — load the pre-built cache
# =========================================================================
def boundary_targets(y):
    """onset[t]=1 where an event starts, offset[t]=1 where it ends."""
    y = np.asarray(y).astype(np.int8)
    onset, offset = np.zeros(len(y), np.float32), np.zeros(len(y), np.float32)
    if len(y) == 0:
        return onset, offset
    prev = np.concatenate(([0], y[:-1]))
    nxt = np.concatenate((y[1:], [0]))
    onset[(y == 1) & (prev == 0)] = 1.0
    offset[(y == 1) & (nxt == 0)] = 1.0
    return onset, offset


def load_cache(cache_dir):
    files = sorted(glob.glob(os.path.join(cache_dir, "*.npz")))
    if not files:
        raise FileNotFoundError(f"no .npz files in {cache_dir}")
    cache = {}
    for f in files:
        m = re.search(r"(SN\d+)", os.path.basename(f))
        if not m:
            continue
        pid = m.group(1)
        with np.load(f, allow_pickle=True) as z:
            if not {"resp", "slow", "y"} <= set(z.files):
                print(f"  [skip] {pid}: missing resp/slow/y (has {list(z.files)})")
                continue
            resp, slow = z["resp"], z["slow"]
            y = np.asarray(z["y"]).astype(np.int8)
            n = min(len(y), resp.shape[0], slow.shape[0])
            d = dict(resp=resp[:n].copy(), slow=slow[:n].copy(), y=y[:n])
            d["feat"] = (np.asarray(z["feat"], np.float32)[:n] if "feat" in z.files
                         else np.zeros((n, 0), np.float32))
            if "onset" in z.files and "offset" in z.files:
                d["onset"] = np.asarray(z["onset"], np.float32)[:n]
                d["offset"] = np.asarray(z["offset"], np.float32)[:n]
            else:
                d["onset"], d["offset"] = boundary_targets(d["y"])
            d["mask_resp"] = (np.asarray(z["mask_resp"], np.float32) if "mask_resp" in z.files
                              else np.ones(resp.shape[1], np.float32))
            d["mask_slow"] = (np.asarray(z["mask_slow"], np.float32) if "mask_slow" in z.files
                              else np.ones(slow.shape[1], np.float32))
        if n >= CHUNK_EPOCHS and y[:n].sum() > 0:
            cache[pid] = d
        else:
            print(f"  [skip] {pid}: {n} epochs, {int(y[:n].sum())} positive")
    return cache


def find_events(seq):
    b = np.asarray(seq).astype(int)
    if b.size == 0:
        return []
    dd = np.diff(np.concatenate(([0], b, [0])))
    return list(zip(np.flatnonzero(dd == 1).tolist(),
                    (np.flatnonzero(dd == -1) - 1).tolist()))


def events_per_hour(y):
    """Events per hour of RECORDING (no sleep staging in this dataset)."""
    return len(find_events(y)) / max(len(y) * WINDOW_SEC / 3600.0, 1e-6)


def severity_of(a):
    return int(np.searchsorted(EPH_BINS, a, side="right"))


CACHE = load_cache(CACHE_DIR)
PIDS = sorted(CACHE, key=lambda p: int(p[2:]))

# layout is READ FROM THE DATA, not assumed
_d0 = CACHE[PIDS[0]]
N_RESP, RESP_SPE = _d0["resp"].shape[1], _d0["resp"].shape[2]
N_SLOW, SLOW_SPE = _d0["slow"].shape[1], _d0["slow"].shape[2]
N_FEAT = _d0["feat"].shape[1]
N_MASK = N_RESP + N_SLOW

COHORT = pd.DataFrame([dict(patient=p, n_epochs=len(CACHE[p]["y"]),
                            apnea_ratio=float(CACHE[p]["y"].mean()),
                            events_per_hour=events_per_hour(CACHE[p]["y"]),
                            severity=SEVERITY_NAMES[severity_of(events_per_hour(CACHE[p]["y"]))])
                       for p in PIDS])
COHORT.to_csv(os.path.join(OUT_DIR, "cohort.csv"), index=False)

n_ep = int(COHORT.n_epochs.sum())
print(f"\n{len(PIDS)} patients | {n_ep:,} epochs ({n_ep*WINDOW_SEC/3600:.0f} h) | "
      f"apnea {100*sum(CACHE[p]['y'].sum() for p in PIDS)/n_ep:.1f}%")
print(f"resp ({N_RESP}, {RESP_SPE}) @ {RESP_SPE/WINDOW_SEC:.0f} Hz | "
      f"slow ({N_SLOW}, {SLOW_SPE}) @ {SLOW_SPE/WINDOW_SEC:.0f} Hz | {N_FEAT} features")
print(COHORT.severity.value_counts().reindex(SEVERITY_NAMES).fillna(0).astype(int).to_string())

## Cell 3 — Dataset

Also verbatim. Chunks of 40 epochs, `gpos` for reassembly, five torch-RNG
augmentations, and `estimate_stats` fitted on the fold's training patients only.

In [ ]:
# =========================================================================
# CELL 3 — dataset
# =========================================================================
class ApneaSeqDataset(Dataset):
    def __init__(self, cache, pids, chunk, stride, stats, augment=False):
        self.cache, self.chunk, self.augment = cache, chunk, augment
        self.pids = [p for p in pids if p in cache]
        (self.rm, self.rs, self.sm, self.ss, self.fm, self.fs) = stats

        self.items, self.offset, self.bounds, labels, off = [], {}, [], [], 0
        for pid in self.pids:
            y = np.asarray(cache[pid]["y"]).astype(np.int8)
            n = len(y)
            self.offset[pid] = off
            self.bounds.append((pid, off, off + n))
            labels.append(y)
            if n >= chunk:
                starts = list(range(0, n - chunk + 1, stride))
                if starts[-1] != n - chunk:
                    starts.append(n - chunk)
            else:
                starts = [0]
            self.items += [(pid, s) for s in starts]
            off += n
        self.total_epochs = off
        self.labels_flat = np.concatenate(labels) if labels else np.zeros(0, np.int8)

    def __len__(self):
        return len(self.items)

    def _slice(self, arr, s, m, spe, nch):
        seg = np.asarray(arr[s:s + m], np.float32).transpose(1, 0, 2).reshape(nch, m * spe)
        if m < self.chunk:
            seg = np.pad(seg, ((0, 0), (0, (self.chunk - m) * spe)))
        return seg

    def __getitem__(self, i):
        pid, s = self.items[i]
        d = self.cache[pid]
        n = len(d["y"])
        if self.augment and n > self.chunk:                    # real time shift
            j = int(torch.randint(-self.chunk // 2, self.chunk // 2 + 1, (1,)).item())
            s = int(np.clip(s + j, 0, n - self.chunk))
        m = min(s + self.chunk, n) - s

        resp = torch.from_numpy((self._slice(d["resp"], s, m, RESP_SPE, N_RESP)
                                 - self.rm[:, None]) / self.rs[:, None])
        slow = torch.from_numpy((self._slice(d["slow"], s, m, SLOW_SPE, N_SLOW)
                                 - self.sm[:, None]) / self.ss[:, None])
        mask = torch.from_numpy(np.concatenate([d["mask_resp"], d["mask_slow"]]
                                               ).astype(np.float32).copy())

        feats = np.zeros((self.chunk, N_FEAT), np.float32)
        if N_FEAT:
            fseg = (np.asarray(d["feat"][s:s + m], np.float32) - self.fm) / self.fs
            feats[:m] = np.clip(fseg, -FEAT_CLIP, FEAT_CLIP)

        y = np.zeros(self.chunk, np.float32); on = np.zeros(self.chunk, np.float32)
        of = np.zeros(self.chunk, np.float32); vl = np.zeros(self.chunk, np.float32)
        y[:m] = d["y"][s:s + m]; on[:m] = d["onset"][s:s + m]
        of[:m] = d["offset"][s:s + m]; vl[:m] = 1.0
        gpos = np.full(self.chunk, -1, np.int64)
        gpos[:m] = np.arange(s, s + m) + self.offset[pid]

        if self.augment:
            # torch RNG only: DataLoader reseeds it per worker, numpy's it does not
            resp = resp + torch.randn_like(resp) * NOISE_STD
            slow = slow + torch.randn_like(slow) * NOISE_STD
            resp = resp * (1 + (torch.rand(N_RESP, 1) * 2 - 1) * SCALE_JITTER)
            slow = slow * (1 + (torch.rand(N_SLOW, 1) * 2 - 1) * SCALE_JITTER)
            dr, ds = torch.rand(N_RESP) < CHANNEL_DROPOUT_P, torch.rand(N_SLOW) < CHANNEL_DROPOUT_P
            if dr.any() and not dr.all():
                resp[dr] = 0.0; mask[:N_RESP][dr] = 0.0      # drop the mask bit too
            if ds.any() and not ds.all():
                slow[ds] = 0.0; mask[N_RESP:][ds] = 0.0
            if float(torch.rand(1)) < TIME_MASK_P:
                w = int(torch.randint(1, max(2, int(TIME_MASK_FRAC * self.chunk)) + 1, (1,)))
                st = int(torch.randint(0, max(1, self.chunk - w), (1,)))
                resp[:, st * RESP_SPE:(st + w) * RESP_SPE] = 0.0   # same wall-clock span
                slow[:, st * SLOW_SPE:(st + w) * SLOW_SPE] = 0.0   # in both streams

        return (resp, slow, mask, torch.from_numpy(feats), torch.from_numpy(y),
                torch.from_numpy(on), torch.from_numpy(of),
                torch.from_numpy(vl), torch.from_numpy(gpos))


def estimate_stats(cache, pids, n_sample=NORM_SAMPLE_EPOCHS):
    """Mean/std per channel and per feature, from TRAINING patients only."""
    rng = np.random.default_rng(SEED)
    pids = [p for p in pids if p in cache]
    budget = max(1, n_sample // max(1, len(pids)))
    rs = rq = ss = sq = fs_ = fq = 0.0
    rc = sc = fc = 0
    for pid in pids:
        d = cache[pid]
        n = d["resp"].shape[0]
        idx = np.sort(rng.choice(n, size=min(budget, n), replace=False))
        r = np.asarray(d["resp"][idx], np.float64); s = np.asarray(d["slow"][idx], np.float64)
        rs = rs + r.sum((0, 2)); rq = rq + (r ** 2).sum((0, 2)); rc += len(idx) * r.shape[2]
        ss = ss + s.sum((0, 2)); sq = sq + (s ** 2).sum((0, 2)); sc += len(idx) * s.shape[2]
        if N_FEAT:
            f = np.asarray(d["feat"][idx], np.float64)
            fs_ = fs_ + f.sum(0); fq = fq + (f ** 2).sum(0); fc += len(idx)
    rc, sc, fc = max(rc, 1), max(sc, 1), max(fc, 1)
    rm, sm = rs / rc, ss / sc
    fm = (fs_ / fc) if N_FEAT else np.zeros(1)
    fsd = np.sqrt(np.clip(fq / fc - fm ** 2, 1e-6, None)) if N_FEAT else np.ones(1)
    return (rm.astype(np.float32),
            np.sqrt(np.clip(rq / rc - rm ** 2, 1e-8, None)).astype(np.float32),
            sm.astype(np.float32),
            np.sqrt(np.clip(sq / sc - sm ** 2, 1e-8, None)).astype(np.float32),
            fm.astype(np.float32), fsd.astype(np.float32))

print("dataset ready")

## Cell 4 — Model and the four losses

The architecture is unchanged. What changes is one function.

### The four losses, in one expression

All four are the same object with two switches:

$$\mathcal{L} = -\frac{1}{\sum_t v_t}\sum_t v_t \Big[\underbrace{w^+}_{\text{switch 1}}\, y_t \underbrace{(1-p_t)^{\gamma}}_{\text{switch 2}} \log p_t \;+\; (1-y_t)\, \underbrace{p_t^{\gamma}}_{\text{switch 2}} \log(1-p_t)\Big]$$

- **Switch 1 — `pos_weight`** $w^+ = \min(n_0/n_1,\,12)$. Multiplies the positive class's
  contribution. Without it, at 24.5% prevalence a model can score 75% accuracy by
  answering "normal" everywhere. Set to 1 in the unweighted arms.
- **Switch 2 — focal** $(1-p_t)^{\gamma}$ with $\gamma = 2$. Down-weights *easy*
  examples: at $p_t = 0.99$ the gradient is scaled by $10^{-4}$, concentrating the loss
  on boundary epochs where the any-overlap labels are noisiest. Set to $\gamma = 0$ in
  the BCE arms, which makes the term exactly 1.

Setting `pos_weight=None` rather than `pos_weight=1.0` in the unweighted arms is
deliberate — it takes the identical code path PyTorch uses when no weighting is
requested, so the BCE arm is a true reference and not "weighted BCE with weight 1".

### Why the cap of 12 matters more in some arms than others

In `weighted_focal` the two switches **multiply**. An uncapped weight of ~3.1 (the
actual $n_0/n_1$ here) is fine, but the cap protects the arm on folds where a patient
mix pushes the ratio higher: focal already down-weights the now-dominant easy negatives,
so an uncapped weight can send the effective positive weight into the hundreds. The cap
is applied in all weighted arms so `weighted_bce` and `weighted_focal` differ *only* in
the focal term — which is the whole point of a factorial design.

### The auxiliary losses stay fixed

$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{arm}} + 0.3\,\mathcal{L}_{\text{burden}} + 0.2\,\mathcal{L}_{\text{boundary}}$$

Burden (chunk apnea fraction) and boundary (onset/offset) always use plain BCE with
their own fixed weights. Only the **main** term varies by arm — otherwise you would be
ablating three things at once and could not attribute the difference.

In [ ]:
# =========================================================================
# CELL 4 — model and the four losses
# =========================================================================
def gn_groups(c, preferred=8):
    for g in range(min(preferred, c), 0, -1):
        if c % g == 0:
            return g
    return 1


class SEBlock1D(nn.Module):
    """Learned per-channel gate: the network decides which features matter."""
    def __init__(self, ch, reduction=SE_REDUCTION):
        super().__init__()
        h = max(4, ch // reduction)
        self.fc = nn.Sequential(nn.Linear(ch, h), nn.GELU(), nn.Linear(h, ch), nn.Sigmoid())

    def forward(self, x):
        b, c, _ = x.shape
        return x * self.fc(x.mean(-1)).view(b, c, 1)


class SEResSepBlock(nn.Module):
    """Dilated depthwise-separable residual block + squeeze-excitation."""
    def __init__(self, ch, kernel=KERNEL, dilation=1, dropout=DROPOUT):
        super().__init__()
        pad = dilation * (kernel // 2)
        self.dw = nn.Conv1d(ch, ch, kernel, padding=pad, dilation=dilation,
                            groups=ch, bias=False)
        self.pw = nn.Conv1d(ch, ch, 1, bias=False)
        self.n1 = nn.GroupNorm(gn_groups(ch), ch)
        self.n2 = nn.GroupNorm(gn_groups(ch), ch)
        self.se, self.act, self.drop = SEBlock1D(ch), nn.GELU(), nn.Dropout(dropout)

    def forward(self, x):
        h = self.act(self.n1(self.dw(x)))
        h = self.se(self.n2(self.pw(h)))
        return self.act(x + self.drop(h))


class StreamEncoder(nn.Module):
    """Signal -> one embedding per 30 s epoch."""
    def __init__(self, in_ch, width, spe, dilations=DILATIONS, stride=STEM_STRIDE):
        super().__init__()
        layers = [nn.Conv1d(in_ch, width, 7, stride=2, padding=3, bias=False),
                  nn.GroupNorm(gn_groups(width), width), nn.GELU()]
        s = 2
        while s < stride:
            layers += [nn.Conv1d(width, width, 5, stride=2, padding=2, bias=False),
                       nn.GroupNorm(gn_groups(width), width), nn.GELU()]
            s *= 2
        self.stem = nn.Sequential(*layers)
        self.body = nn.Sequential(*[SEResSepBlock(width, dilation=d) for d in dilations])
        self.spe_out = max(1, spe // s)
        self.out_dim = 2 * width                      # mean + max pooling

    def forward(self, x):
        h = self.body(self.stem(x))
        b, w, t = h.shape
        n_ep = max(1, t // self.spe_out)
        h = h[:, :, :n_ep * self.spe_out].view(b, w, n_ep, self.spe_out)
        # mean AND max: any-overlap labels are effectively a max over the epoch
        return torch.cat([h.mean(-1), h.amax(-1)], dim=1).transpose(1, 2)


class ApneaSegmenter(nn.Module):
    def __init__(self):
        super().__init__()
        self.resp_enc = StreamEncoder(N_RESP, WIDTH, RESP_SPE)
        self.slow_enc = StreamEncoder(N_SLOW, WIDTH, SLOW_SPE)
        gru_in = self.resp_enc.out_dim + self.slow_enc.out_dim + N_FEAT
        self.temporal = nn.GRU(gru_in, GRU_HIDDEN, num_layers=GRU_LAYERS,
                               batch_first=True, bidirectional=True,
                               dropout=DROPOUT if GRU_LAYERS > 1 else 0.0)
        self.norm, self.drop = nn.LayerNorm(2 * GRU_HIDDEN), nn.Dropout(DROPOUT)
        # the mask joins HERE, not in the encoder: constant along time, and
        # sensor availability is a patient fingerprint = a leakage path
        self.head = nn.Linear(2 * GRU_HIDDEN + N_MASK, 3 if USE_BOUNDARY_HEAD else 1)
        self.burden = nn.Sequential(nn.Linear(2 * GRU_HIDDEN, GRU_HIDDEN), nn.GELU(),
                                    nn.Dropout(DROPOUT), nn.Linear(GRU_HIDDEN, 1)
                                    ) if USE_BURDEN_HEAD else None

    def forward(self, resp, slow, mask, feats):
        h = torch.cat([self.resp_enc(resp), self.slow_enc(slow)], dim=-1)
        if N_FEAT:
            h = torch.cat([h, feats], dim=-1)
        h, _ = self.temporal(h)
        h = self.drop(self.norm(h))
        m = mask[:, None, :].expand(-1, h.shape[1], -1)
        out = self.head(torch.cat([h, m], dim=-1))
        apnea = out[..., 0]
        onset = out[..., 1] if USE_BOUNDARY_HEAD else None
        offset = out[..., 2] if USE_BOUNDARY_HEAD else None
        burden = self.burden(h.mean(1)).squeeze(-1) if self.burden is not None else None
        return apnea, onset, offset, burden


# ---------------- the 2x2 loss family ------------------------------------
def apnea_loss(logits, targets, valid, pos_weight, loss_type):
    """bce | weighted_bce | focal | weighted_focal, as two switches."""
    pw = pos_weight if loss_type.startswith("weighted") else None   # switch 1
    loss = F.binary_cross_entropy_with_logits(logits, targets,
                                              pos_weight=pw, reduction="none")
    if "focal" in loss_type:                                        # switch 2
        p = torch.sigmoid(logits)
        pt = p * targets + (1 - p) * (1 - targets)
        loss = loss * (1 - pt).clamp(min=1e-6) ** FOCAL_GAMMA
    return (loss * valid).sum() / valid.sum().clamp(min=1.0)


def total_loss(apnea, onset, offset, burden, y, y_on, y_off, valid,
               pos_weight, bnd_weight, loss_type):
    """Only the MAIN term varies by arm; the auxiliaries are fixed across arms."""
    main = apnea_loss(apnea, y, valid, pos_weight, loss_type)
    total, parts = main, {"main": float(main.detach())}
    if burden is not None:
        tgt = (y * valid).sum(1) / valid.sum(1).clamp(min=1.0)
        aux = F.binary_cross_entropy_with_logits(burden, tgt)
        total = total + BURDEN_WEIGHT * aux
        parts["burden"] = float(aux.detach())
    if onset is not None:
        b1 = F.binary_cross_entropy_with_logits(onset, y_on, pos_weight=bnd_weight,
                                                reduction="none")
        b2 = F.binary_cross_entropy_with_logits(offset, y_off, pos_weight=bnd_weight,
                                                reduction="none")
        bnd = ((b1 + b2) * valid).sum() / valid.sum().clamp(min=1.0) / 2.0
        total = total + BOUNDARY_WEIGHT * bnd
        parts["boundary"] = float(bnd.detach())
    return total, parts


_m = ApneaSegmenter()
print(f"ApneaSegmenter: {sum(p.numel() for p in _m.parameters()):,} parameters")
print(f"receptive field: {1 + 2*(KERNEL//2)*sum(DILATIONS)} taps "
      f"= {(1 + 2*(KERNEL//2)*sum(DILATIONS)) / (RESP_SPE/WINDOW_SEC/STEM_STRIDE):.1f} s")
del _m

# sanity check: all four arms produce a finite, distinct loss on the same logits
_lg = torch.randn(2, 40); _ty = (torch.rand(2, 40) < 0.25).float()
_v = torch.ones(2, 40); _pw = torch.tensor([3.1])
print("\\n  arm             loss on identical random logits")
for a in ARMS:
    print(f"  {a:16s} {float(apnea_loss(_lg, _ty, _v, _pw, a)):.4f}")
del _lg, _ty, _v, _pw

## Cell 5 — Train one fold, for a given arm

`train_fold(test_pid, loss_type)` — one extra argument compared with the LOPO notebook,
threaded down into `total_loss`.

Everything else is held fixed on purpose:

- **`make_fold` is deterministic** and does not depend on the arm, so arm A and arm B
  test on the same patient with the same 76 training and 12 validation patients. This is
  what makes the per-patient results **paired**, and paired data supports a much more
  sensitive test than comparing two independent means.
- **`torch.manual_seed(SEED)` is reset at the top of every fold**, so all four arms start
  from *identical* initial weights. Any divergence is the loss, not initialization.
- `pos_weight` is computed and passed in every arm. `apnea_loss` simply ignores it in the
  unweighted arms — this keeps one code path instead of two.

### The threshold is re-tuned per arm

Each arm gets its own F1-optimal threshold from its own validation predictions. This is
essential and easy to get wrong: focal loss systematically shifts probabilities downward,
so scoring a focal model at BCE's threshold would measure the mismatch rather than the
loss. Every arm is evaluated at *its own* best operating point.

In [ ]:
# =========================================================================
# CELL 5 — one fold, one arm
# =========================================================================
def make_fold(test_pid):
    """Deterministic and arm-independent -> the four arms are exactly paired."""
    others = [p for p in PIDS if p != test_pid]
    ranks = np.argsort([events_per_hour(CACHE[p]["y"]) for p in others], kind="stable")
    ordered = [others[i] for i in ranks]
    vi = sorted(set(np.linspace(0, len(ordered) - 1, VAL_K).round().astype(int).tolist()))
    return ([p for i, p in enumerate(ordered) if i not in set(vi)],
            [ordered[i] for i in vi], [test_pid])


def best_f1_threshold(labels, probs):
    """F1-optimal cut. Only ever called on VALIDATION."""
    labels, probs = np.asarray(labels), np.asarray(probs)
    if labels.size == 0 or len(np.unique(labels)) < 2:
        return 0.5, 0.0
    prec, rec, thr = precision_recall_curve(labels, probs)
    if len(thr) == 0:
        return 0.5, 0.0
    f1s = 2 * prec * rec / (prec + rec + 1e-9)
    i = int(np.nanargmax(f1s[:-1]))
    return float(thr[i]), float(f1s[i])


def run_pass(model, loader, pos_weight, bnd_weight, optimizer, scaler,
             scheduler, train_mode, loss_type, total_epochs=None):
    model.train(mode=train_mode)
    loss_sum = seen = 0.0
    if not train_mode:
        psum = np.zeros(total_epochs); pcnt = np.zeros(total_epochs)

    for resp, slow, mask, feats, y, y_on, y_off, valid, gpos in loader:
        resp, slow = resp.to(DEVICE, non_blocking=True), slow.to(DEVICE, non_blocking=True)
        mask, feats = mask.to(DEVICE), feats.to(DEVICE)
        y, y_on, y_off, valid = (y.to(DEVICE), y_on.to(DEVICE),
                                 y_off.to(DEVICE), valid.to(DEVICE))
        with torch.set_grad_enabled(train_mode):
            with torch.amp.autocast("cuda", enabled=(DEVICE.type == "cuda")):
                apnea, onset, offset, burden = model(resp, slow, mask, feats)
                loss, _ = total_loss(apnea, onset, offset, burden, y, y_on, y_off,
                                     valid, pos_weight, bnd_weight, loss_type)
            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)              # must precede clipping
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer); scaler.update()
                if scheduler is not None:
                    scheduler.step()

        nv = float(valid.sum().item())
        loss_sum += float(loss.item()) * nv; seen += nv
        if not train_mode:
            p = torch.sigmoid(apnea.float()).detach().cpu().numpy().ravel()
            g = gpos.numpy().ravel()
            keep = (valid.detach().cpu().numpy().ravel() > 0.5) & (g >= 0)
            np.add.at(psum, g[keep], p[keep])           # unbuffered: handles overlap
            np.add.at(pcnt, g[keep], 1.0)

    avg = loss_sum / max(seen, 1.0)
    return (avg, None) if train_mode else (avg, psum / np.maximum(pcnt, 1.0))


def train_fold(test_pid, loss_type, verbose=True):
    tr_p, va_p, te_p = make_fold(test_pid)
    random.seed(SEED); np.random.seed(SEED)
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)   # identical init per arm

    stats = estimate_stats(CACHE, tr_p)                          # TRAIN ONLY
    tr = ApneaSeqDataset(CACHE, tr_p, CHUNK_EPOCHS, TRAIN_STRIDE, stats, augment=AUGMENT)
    va = ApneaSeqDataset(CACHE, va_p, CHUNK_EPOCHS, EVAL_STRIDE, stats)
    te = ApneaSeqDataset(CACHE, te_p, CHUNK_EPOCHS, EVAL_STRIDE, stats)

    mk = lambda ds, sh, dl: DataLoader(ds, batch_size=BATCH_SIZE, shuffle=sh,
                                       num_workers=NUM_WORKERS, drop_last=dl,
                                       pin_memory=(DEVICE.type == "cuda"),
                                       persistent_workers=(NUM_WORKERS > 0))
    tl = mk(tr, True, len(tr) >= 2 * BATCH_SIZE)
    vl, sl = mk(va, False, False), mk(te, False, False)

    n1 = int(tr.labels_flat.sum()); n0 = len(tr.labels_flat) - n1
    pw = float(min(n0 / max(1, n1), POS_WEIGHT_CAP))
    pos_weight = torch.tensor([pw], dtype=torch.float32, device=DEVICE)
    bnd_weight = torch.tensor([BOUNDARY_POS_WEIGHT], dtype=torch.float32, device=DEVICE)

    model = ApneaSegmenter().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR,
                                                total_steps=EPOCHS * max(1, len(tl)),
                                                pct_start=0.25)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

    best_f1, best_state, best_thr, bad, hist = -1.0, None, 0.5, 0, []
    for ep in range(1, EPOCHS + 1):
        t0 = time.time()
        trl, _ = run_pass(model, tl, pos_weight, bnd_weight, opt, scaler, sched,
                          True, loss_type)
        vll, vp = run_pass(model, vl, pos_weight, bnd_weight, opt, scaler, None,
                           False, loss_type, va.total_epochs)
        thr, f1 = best_f1_threshold(va.labels_flat, vp)
        hist.append(dict(epoch=ep, train_loss=float(trl), val_loss=float(vll),
                         val_f1=float(f1), threshold=float(thr)))
        if verbose:
            print(f"      ep {ep:2d}/{EPOCHS} train {trl:.4f} val {vll:.4f} "
                  f"val_F1 {f1:.4f} (thr {thr:.2f}) {time.time()-t0:.0f}s")
        if f1 > best_f1:
            best_f1, best_thr, bad = f1, thr, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                if verbose:
                    print(f"      early stop at epoch {ep}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    _, test_probs = run_pass(model, sl, pos_weight, bnd_weight, opt, scaler, None,
                             False, loss_type, te.total_epochs)

    del model, best_state
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return dict(probs=test_probs.astype(np.float32), labels=te.labels_flat.astype(np.int8),
                threshold=best_thr, val_f1=best_f1, pos_weight=pw, hist=hist)

print("training engine ready |", len(ARMS), "arms")

## Cell 6 — Per-patient metrics

Same definitions as the LOPO notebook, with the F1 fix described at the top.

### `NaN` vs `0.0`, stated precisely

| Situation | Correct value | Why |
|---|---|---|
| patient has **no positive epochs** | `NaN` | recall is 0/0 — the metric does not exist |
| model predicts **no positives** on a patient that has some | **0.0** | precision is 0/0 but F1 is genuinely 0 — the model found nothing |
| model predicts positives, none correct | **0.0** | a real, measurable failure |

Recording the last two as `NaN` silently deletes the model's worst patients from the
average. The rule below: `NaN` only when the *labels* make a metric undefined; `0.0`
whenever the model simply failed.

### Metrics collected, grouped by the question

- **At the operating point** — accuracy, precision, recall/sensitivity, specificity, NPV,
  F1, balanced accuracy, MCC, Cohen's κ
- **Threshold-free** — ROC-AUC, PR-AUC
- **Calibration** — Brier, ECE
- **Clinical** — events per hour (true and predicted), severity band

MCC is worth watching alongside F1: F1 ignores true negatives entirely, MCC uses all four
cells of the confusion matrix, so a model that wins on F1 but loses on MCC is buying
recall with false positives.

In [ ]:
# =========================================================================
# CELL 6 — per-patient metrics
# =========================================================================
def expected_calibration_error(labels, probs, n_bins=N_CALIB_BINS):
    labels, probs = np.asarray(labels).astype(int), np.asarray(probs, float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece, N = 0.0, len(probs)
    for i in range(n_bins):
        hi = probs <= edges[i + 1] if i == n_bins - 1 else probs < edges[i + 1]
        m = (probs >= edges[i]) & hi
        if m.any():
            ece += (m.sum() / N) * abs(labels[m].mean() - probs[m].mean())
    return float(ece)


def patient_metrics(labels, probs, thr):
    """NaN only when the LABELS make a metric undefined; 0.0 when the model failed."""
    labels = np.asarray(labels).astype(int)
    pred = (np.asarray(probs) >= thr).astype(int)
    has_pos, has_neg = labels.sum() > 0, (labels == 0).sum() > 0
    two = has_pos and has_neg
    tn, fp, fn, tp = confusion_matrix(labels, pred, labels=[0, 1]).ravel()

    sens = (tp / (tp + fn)) if has_pos else np.nan
    spec = (tn / (tn + fp)) if has_neg else np.nan
    # precision: 0/0 with true positives present is a real failure -> 0.0
    prec = (tp / (tp + fp)) if (tp + fp) else (0.0 if has_pos else np.nan)
    if not has_pos:
        f1 = np.nan
    elif (prec + sens) == 0:
        f1 = 0.0                       # <-- the fix
    else:
        f1 = 2 * prec * sens / (prec + sens)

    return dict(
        n_epochs=len(labels), n_apnea=int(labels.sum()),
        prevalence=float(labels.mean()), threshold=float(thr),
        accuracy=accuracy_score(labels, pred),
        precision=prec, recall=sens, sensitivity=sens, specificity=spec,
        npv=(tn / (tn + fn)) if (tn + fn) else np.nan,
        f1=f1,
        balanced_accuracy=(sens + spec) / 2 if np.isfinite(sens) and np.isfinite(spec) else np.nan,
        mcc=matthews_corrcoef(labels, pred) if two else np.nan,
        cohen_kappa=cohen_kappa_score(labels, pred) if two else np.nan,
        roc_auc=roc_auc_score(labels, probs) if two else np.nan,
        pr_auc=average_precision_score(labels, probs) if two else np.nan,
        brier=brier_score_loss(labels, np.clip(probs, 0, 1)),
        ece=expected_calibration_error(labels, probs),
        tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp),
        true_eph=events_per_hour(labels), pred_eph=events_per_hour(pred),
        severity=SEVERITY_NAMES[severity_of(events_per_hour(labels))],
        n_true_events=len(find_events(labels)), n_pred_events=len(find_events(pred)))


METRIC_COLS = ["accuracy", "precision", "recall", "sensitivity", "specificity", "npv",
               "f1", "balanced_accuracy", "mcc", "cohen_kappa", "roc_auc", "pr_auc",
               "brier", "ece"]
print("metrics ready |", len(METRIC_COLS), "averaged metrics")

## Cell 7 — Events-per-hour agreement, including R²

Computed **across** folds — a single test patient gives one point, so correlation and
agreement need all 89 pairs pooled.

### Two different R², and they are not the same number

**Squared Pearson correlation** $r^2$ — how much of the variance in true events per hour
is explained by a *best-fit line* through the predictions. It is invariant to any
systematic offset or scaling: a model that predicts exactly half the true rate every time
still scores $r^2 = 1.0$.

**Coefficient of determination** $R^2 = 1 - \dfrac{\sum(\hat y - y)^2}{\sum(y - \bar y)^2}$
— how much better the predictions are than simply guessing the cohort mean, *on the
identity line*. Bias and scale errors are penalised, and it can go negative.

Your last run had bias −1.065/h, which is exactly the situation where these two diverge.
**Report $R^2$, not $r^2$** — the coefficient of determination is what tells a clinician
whether the predicted number is usable, and quoting $r^2$ where $R^2$ is much lower is one
of the more common ways a methods section overstates agreement. Both are computed, and the
gap between them is printed.

Also reported: MAE, bias, 95% limits of agreement (bias ± 1.96 SD), severity accuracy and
quadratic-weighted κ. Quadratic weighting penalises a normal↔severe confusion 9× more than
normal↔mild, matching the clinical cost.

In [ ]:
# =========================================================================
# CELL 7 — events-per-hour agreement
# =========================================================================
def eph_agreement(df):
    """Cohort-level agreement between true and predicted events per hour."""
    d = df[["true_eph", "pred_eph"]].dropna()
    out = dict(n=len(d))
    if len(d) < 3:
        return {**out, **{k: np.nan for k in
                ("eph_pearson_r", "eph_r2_pearson", "eph_r2_determination", "eph_spearman",
                 "eph_mae", "eph_rmse", "eph_bias", "eph_loa_lo", "eph_loa_hi",
                 "severity_acc", "severity_kappa_w")}}
    t, p = d.true_eph.values, d.pred_eph.values
    diff = p - t
    r = float(pearsonr(t, p)[0])
    ss_res = float(((p - t) ** 2).sum())
    ss_tot = float(((t - t.mean()) ** 2).sum())
    ts = np.array([severity_of(v) for v in t])
    ps = np.array([severity_of(v) for v in p])
    out.update(eph_pearson_r=r,
               eph_r2_pearson=r ** 2,                                  # line-of-best-fit
               eph_r2_determination=1 - ss_res / max(ss_tot, 1e-9),    # identity line
               eph_spearman=float(spearmanr(t, p)[0]),
               eph_mae=float(np.abs(diff).mean()),
               eph_rmse=float(np.sqrt((diff ** 2).mean())),
               eph_bias=float(diff.mean()),
               eph_loa_lo=float(diff.mean() - 1.96 * diff.std(ddof=1)),
               eph_loa_hi=float(diff.mean() + 1.96 * diff.std(ddof=1)),
               severity_acc=float((ts == ps).mean()),
               severity_kappa_w=float(cohen_kappa_score(ts, ps, weights="quadratic"))
                                 if len(np.unique(ts)) > 1 else np.nan)
    return out

print("events-per-hour agreement ready")

## Cell 8 — The ablation loop

For each arm, for each of the 89 patients: train, predict, score, save.

**Fold files are per arm** (`{arm}/folds/{pid}.npz`) so nothing collides and each arm
resumes independently. A corrupt or truncated file is detected, deleted and retrained
rather than aborting the run — the failure mode you hit before.

### Importing already-trained folds

`SEED_FOLDS_FROM` copies your existing `weighted_focal` folds instead of retraining them.
Before copying, `_config_matches` checks that each file has the expected keys and a
plausible probability array. It **cannot** verify that the source run used the same
`EPOCHS`, `VAL_K` or `EVAL_STRIDE` — no record of those is stored in the fold file — so
confirm that yourself. If in doubt, set `SEED_FOLDS_FROM = {}` and pay the 6 hours; a
contaminated arm invalidates the whole comparison.

The running mean printed per fold is a live sanity check. If an arm's running F1 sits far
from the others after 20 folds, stop and look rather than waiting 6 hours.

In [ ]:
# =========================================================================
# CELL 8 — ablation loop
# =========================================================================
def fold_path(arm, pid):
    return os.path.join(arm_dir(arm, "folds"), f"{pid}.npz")


def _load_fold(fp):
    """Cached fold, or None if missing/corrupt (corrupt files are removed)."""
    if not os.path.exists(fp):
        return None
    try:
        with np.load(fp) as z:
            if not {"probs", "labels", "threshold", "val_f1"} <= set(z.files):
                raise ValueError("missing keys")
            r = dict(probs=z["probs"], labels=z["labels"],
                     threshold=float(z["threshold"]), val_f1=float(z["val_f1"]),
                     hist=json.loads(str(z["hist"])) if "hist" in z.files else [])
        if len(r["probs"]) != len(r["labels"]) or len(r["probs"]) == 0:
            raise ValueError("length mismatch")
        return r
    except Exception as e:
        print(f"      [warn] {os.path.basename(fp)} unreadable ({type(e).__name__}) "
              f"-- retraining")
        os.remove(fp)
        return None


def _save_fold(fp, r):
    tmp = fp + ".tmp.npz"
    with open(tmp, "wb") as fh:
        np.savez(fh, probs=r["probs"], labels=r["labels"], threshold=r["threshold"],
                 val_f1=r["val_f1"], hist=json.dumps(r.get("hist", [])))
        fh.flush(); os.fsync(fh.fileno())      # surfaces ENOSPC instead of a hollow file
    os.replace(tmp, fp)


def import_seed_folds():
    """Copy already-trained folds in, so they are not retrained."""
    for arm, src in SEED_FOLDS_FROM.items():
        if arm not in ARMS or not os.path.isdir(src):
            continue
        n = 0
        for pid in PIDS:
            dst = fold_path(arm, pid)
            s = os.path.join(src, f"{pid}.npz")
            if os.path.exists(s) and not os.path.exists(dst) and _load_fold(s) is not None:
                shutil.copy2(s, dst); n += 1
        if n:
            print(f"  imported {n} '{arm}' folds from {src}")
            print(f"  [!] verify that run used EPOCHS={EPOCHS}, VAL_K={VAL_K}, "
                  f"EVAL_STRIDE={EVAL_STRIDE}, SEED={SEED}")


def run_arm(arm, pids=None, max_folds=MAX_FOLDS, resume=RESUME, verbose=False):
    pids = list(pids or PIDS)
    if max_folds:
        pids = pids[:max_folds]
    rows, hist_rows, t0 = [], [], time.time()

    for i, pid in enumerate(pids, 1):
        fp = fold_path(arm, pid)
        r = _load_fold(fp) if resume else None
        tag = "cached"
        if r is None:
            print(f"    [{arm}] fold {i}/{len(pids)} — {pid} "
                  f"({len(CACHE[pid]['y'])} ep, {100*CACHE[pid]['y'].mean():.1f}% apnea)")
            r = train_fold(pid, arm, verbose=verbose)
            _save_fold(fp, r)
            tag = "trained"

        m = patient_metrics(r["labels"], r["probs"], r["threshold"])
        m.update(patient=pid, arm=arm, val_f1=r["val_f1"], source=tag)
        rows.append(m)
        for h in r.get("hist", []):
            hist_rows.append(dict(arm=arm, patient=pid, **h))

        done = pd.DataFrame(rows)
        done.to_csv(arm_dir(arm, "per_patient.csv"), index=False)
        if i % 5 == 0 or i == len(pids) or tag == "trained":
            print(f"    [{arm} {i}/{len(pids)} {tag}] {pid}: F1 {m['f1']:.4f} "
                  f"AUC {m['roc_auc']:.4f} | running mean F1 {np.nanmean(done.f1):.4f} "
                  f"| {(time.time()-t0)/60:.1f} min")

    if hist_rows:
        pd.DataFrame(hist_rows).to_csv(arm_dir(arm, "loss_history.csv"), index=False)
    return pd.DataFrame(rows)


def run_ablation(arms=None):
    import_seed_folds()
    arms = arms or ARMS_TO_RUN
    out = {}
    for arm in arms:
        done = len(glob.glob(os.path.join(arm_dir(arm, "folds"), "*.npz")))
        todo = (len(PIDS) if MAX_FOLDS is None else min(MAX_FOLDS, len(PIDS))) - done
        print(f"\n{'='*78}\n  ARM: {ARM_LABEL[arm]}  ({arm})\n"
              f"  {done} folds cached, {max(todo,0)} to train "
              f"(~{max(todo,0)*4/60:.1f} h on GPU)\n{'='*78}")
        out[arm] = run_arm(arm)
    return out


_pending = sum(max((len(PIDS) if MAX_FOLDS is None else MAX_FOLDS)
                   - len(glob.glob(os.path.join(arm_dir(a, "folds"), "*.npz"))), 0)
               for a in ARMS_TO_RUN)
print(f"ready: {len(ARMS_TO_RUN)} arms x {len(PIDS)} patients")
print(f"{_pending} folds still to train  (~{_pending*4/60:.1f} h on GPU)")

## Cell 9 — Run

The long cell. Interruptible: every finished fold is on disk and skipped on re-run.

In [ ]:
# =========================================================================
# CELL 9 — RUN
# =========================================================================
T0 = time.time()
RESULTS = run_ablation()
print(f"\nablation complete in {(time.time()-T0)/3600:.2f} h")
for a, df in RESULTS.items():
    print(f"  {ARM_LABEL[a]:16s} {len(df)} folds, mean F1 {np.nanmean(df.f1):.4f}")

## Cell 10 — Per-arm summaries and the comparison table

Two levels of output.

**Per arm** — `{arm}/summary.csv`, the same shape as `lopo_summary.csv`: one row per
metric with mean, SD, SEM, median, min, max, `n_valid`, and the pooled value.

**Across arms** — `comparison/csv/`:

| File | Contents |
|---|---|
| `metric_comparison.csv` | the headline: one row per metric, one column per arm (mean ± SD) |
| `metric_comparison_wide.csv` | flat numeric version for plotting or a LaTeX table |
| `arm_headline.csv` | one row per arm, all requested metrics side by side |
| `eph_agreement.csv` | Pearson r, r², **R²**, MAE, RMSE, bias, LoA, severity κ_w per arm |
| `severity_strata_by_arm.csv` | F1 / AUC / κ per severity band per arm |
| `per_patient_all_arms.csv` | 89 × 4 rows — the raw material for the paired tests |

### On the pooled column

Pooled accuracy, F1, sensitivity and so on are sound — each patient is binarized at its
own fold threshold before concatenation. Pooled **AUROC, PR-AUC, Brier and ECE** are not:
they mix probabilities from 89 differently-calibrated models. They are computed for
completeness and as a diagnostic (a large macro–pooled gap means the folds disagree about
where probabilities sit), but **report the macro values**.

In [ ]:
# =========================================================================
# CELL 10 — summaries and comparison tables
# =========================================================================
def load_arm(arm):
    p = arm_dir(arm, "per_patient.csv")
    return pd.read_csv(p) if os.path.exists(p) else pd.DataFrame()


def pooled_for_arm(arm, df):
    """All test patients concatenated, each binarized at its own fold threshold."""
    L, P, PR = [], [], []
    for pid in df.patient:
        r = _load_fold(fold_path(arm, pid))
        if r is None:
            continue
        L.append(r["labels"]); P.append(r["probs"])
        PR.append((r["probs"] >= r["threshold"]).astype(int))
    lab, prob, pred = np.concatenate(L), np.concatenate(P), np.concatenate(PR)
    tn, fp, fn, tp = confusion_matrix(lab, pred, labels=[0, 1]).ravel()
    sens, spec = tp / (tp + fn), tn / (tn + fp)
    d = dict(accuracy=accuracy_score(lab, pred),
             precision=precision_score(lab, pred, zero_division=0),
             recall=sens, sensitivity=sens, specificity=spec,
             npv=tn / max(tn + fn, 1), f1=f1_score(lab, pred, zero_division=0),
             balanced_accuracy=(sens + spec) / 2,
             mcc=matthews_corrcoef(lab, pred), cohen_kappa=cohen_kappa_score(lab, pred),
             roc_auc=roc_auc_score(lab, prob), pr_auc=average_precision_score(lab, prob),
             brier=brier_score_loss(lab, np.clip(prob, 0, 1)),
             ece=expected_calibration_error(lab, prob))
    return d, lab, prob, pred, np.array([[tn, fp], [fn, tp]])


def summarize(df):
    rows = []
    for c in METRIC_COLS:
        v = df[c].astype(float).values
        ok = v[np.isfinite(v)]
        rows.append(dict(metric=c,
                         mean=ok.mean() if len(ok) else np.nan,
                         std=ok.std(ddof=1) if len(ok) > 1 else np.nan,
                         sem=ok.std(ddof=1) / np.sqrt(len(ok)) if len(ok) > 1 else np.nan,
                         median=np.median(ok) if len(ok) else np.nan,
                         min=ok.min() if len(ok) else np.nan,
                         max=ok.max() if len(ok) else np.nan,
                         n_valid=len(ok), n_undefined=len(v) - len(ok)))
    return pd.DataFrame(rows)


ARM_DF, SUMMARY, POOLED, CONFUSION, POOLED_RAW, EPH = {}, {}, {}, {}, {}, {}
for arm in ARMS:
    df = load_arm(arm)
    if not len(df):
        print(f"  [skip] {arm}: no results yet")
        continue
    ARM_DF[arm] = df
    s = summarize(df)
    pooled, lab, prob, pred, cm = pooled_for_arm(arm, df)
    s["pooled"] = s.metric.map(pooled)
    s["pooled_minus_mean"] = s.pooled - s["mean"]
    s.to_csv(arm_dir(arm, "summary.csv"), index=False)
    SUMMARY[arm], POOLED[arm], CONFUSION[arm] = s, pooled, cm
    POOLED_RAW[arm] = dict(labels=lab, probs=prob, pred=pred)
    EPH[arm] = eph_agreement(df)

ARMS_DONE = list(ARM_DF)
ALL = pd.concat([ARM_DF[a].assign(arm=a) for a in ARMS_DONE], ignore_index=True)
ALL.to_csv(os.path.join(CMP_CSV, "per_patient_all_arms.csv"), index=False)

# ---- headline: one row per arm ------------------------------------------
HEAD_COLS = ["accuracy", "precision", "recall", "sensitivity", "specificity", "npv",
             "f1", "balanced_accuracy", "mcc", "cohen_kappa", "roc_auc", "pr_auc",
             "brier", "ece"]
rows = []
for a in ARMS_DONE:
    r = dict(arm=a, label=ARM_LABEL[a], n_folds=len(ARM_DF[a]))
    for m in HEAD_COLS:
        r[m] = float(np.nanmean(ARM_DF[a][m]))
        r[f"{m}_sd"] = float(np.nanstd(ARM_DF[a][m], ddof=1))
    r.update({k: v for k, v in EPH[a].items() if k != "n"})
    r["mean_threshold"] = float(ARM_DF[a].threshold.mean())
    r["mean_val_f1"] = float(ARM_DF[a].val_f1.mean())
    rows.append(r)
HEADLINE = pd.DataFrame(rows)
HEADLINE.to_csv(os.path.join(CMP_CSV, "arm_headline.csv"), index=False)

# ---- the requested comparison table -------------------------------------
REQUESTED = [("accuracy", "Accuracy"), ("precision", "Precision"),
             ("recall", "Recall"), ("sensitivity", "Sensitivity"),
             ("specificity", "Specificity"), ("f1", "F1 score"),
             ("mcc", "MCC"), ("cohen_kappa", "Cohen kappa"),
             ("roc_auc", "ROC AUC"), ("pr_auc", "PR AUC"),
             ("balanced_accuracy", "Balanced accuracy"),
             ("brier", "Brier"), ("ece", "ECE")]
EPH_REQUESTED = [("eph_pearson_r", "Events/h Pearson r"),
                 ("eph_r2_determination", "Events/h R2 (determination)"),
                 ("eph_r2_pearson", "Events/h r2 (Pearson squared)"),
                 ("eph_mae", "Events/h MAE"), ("eph_bias", "Events/h bias"),
                 ("severity_acc", "Severity accuracy"),
                 ("severity_kappa_w", "Severity kappa_w")]

rows = []
for key, label in REQUESTED:
    r = {"metric": label, "key": key}
    for a in ARMS_DONE:
        v = ARM_DF[a][key].astype(float)
        r[ARM_LABEL[a]] = f"{np.nanmean(v):.4f} ± {np.nanstd(v, ddof=1):.4f}"
    rows.append(r)
for key, label in EPH_REQUESTED:
    r = {"metric": label, "key": key}
    for a in ARMS_DONE:
        r[ARM_LABEL[a]] = f"{EPH[a][key]:.4f}"
    rows.append(r)
COMPARISON = pd.DataFrame(rows)
COMPARISON.to_csv(os.path.join(CMP_CSV, "metric_comparison.csv"), index=False)

WIDE = pd.DataFrame({"metric": [l for _, l in REQUESTED + EPH_REQUESTED],
                     "key": [k for k, _ in REQUESTED + EPH_REQUESTED]})
for a in ARMS_DONE:
    WIDE[a] = ([float(np.nanmean(ARM_DF[a][k])) for k, _ in REQUESTED]
               + [float(EPH[a][k]) for k, _ in EPH_REQUESTED])
def _best_arm(row):
    v = row[ARMS_DONE].astype(float)
    if not np.isfinite(v).any():          # metric undefined for every arm
        return "n/a"
    if row["key"] in ("brier", "ece", "eph_mae"):
        return v.idxmin()                 # lower is better
    if row["key"] == "eph_bias":
        return v.abs().idxmin()           # closest to zero wins
    return v.idxmax()

WIDE["best_arm"] = WIDE.apply(_best_arm, axis=1)
WIDE.to_csv(os.path.join(CMP_CSV, "metric_comparison_wide.csv"), index=False)
pd.DataFrame([{"arm": a, "label": ARM_LABEL[a], **EPH[a]} for a in ARMS_DONE]) \
  .to_csv(os.path.join(CMP_CSV, "eph_agreement.csv"), index=False)

# ---- severity strata by arm ---------------------------------------------
_ord = {n: i for i, n in enumerate(SEVERITY_NAMES)}
STRATA = (ALL.groupby(["arm", "severity"])
            .agg(n_patients=("patient", "count"), prevalence=("prevalence", "mean"),
                 f1=("f1", "mean"), sensitivity=("sensitivity", "mean"),
                 specificity=("specificity", "mean"), roc_auc=("roc_auc", "mean"),
                 pr_auc=("pr_auc", "mean"), cohen_kappa=("cohen_kappa", "mean"))
            .reset_index().sort_values(["arm", "severity"], key=lambda s: s.map(_ord)
                                       if s.name == "severity" else s))
STRATA.to_csv(os.path.join(CMP_CSV, "severity_strata_by_arm.csv"), index=False)

print("=" * 110)
print(f"  LOSS ABLATION — {len(ARMS_DONE)} arms x {len(ARM_DF[ARMS_DONE[0]])} "
      f"leave-one-patient-out folds")
print("=" * 110)
print(COMPARISON[["metric"] + [ARM_LABEL[a] for a in ARMS_DONE]].to_string(index=False))

print("\n---- BEST ARM PER METRIC ----")
print(WIDE[["metric"] + ARMS_DONE + ["best_arm"]]
      .to_string(index=False, float_format=lambda v: f"{v:9.4f}"))
print(f"\nwins: {WIDE[WIDE.best_arm != 'n/a'].best_arm.value_counts().to_dict()}")

## Cell 11 — Overall confusion matrices

One pooled confusion matrix per arm, over all 85,288 test epochs — every patient
binarized at its own fold's threshold, then concatenated.

Each is shown twice because the two normalizations answer different questions:

- **Counts** — the raw scale. Useful for spotting that FP and FN are not remotely
  symmetric in cost here.
- **Row-normalized** — each row sums to 1, so the diagonal reads directly as
  **sensitivity** (bottom-right) and **specificity** (top-left), independent of the 3:1
  class imbalance. This is the one to read when comparing arms.

At 24.5% prevalence the raw matrix is dominated by true negatives, so an arm can look
excellent in counts while missing a third of all events. Row normalization removes that
illusion.

`confusion_matrices.csv` records TN/FP/FN/TP per arm alongside the derived rates, so the
numbers in the figure are reproducible from the CSV alone.

In [ ]:
# =========================================================================
# CELL 11 — pooled confusion matrices
# =========================================================================
rows = []
for a in ARMS_DONE:
    tn, fp = CONFUSION[a][0]
    fn, tp = CONFUSION[a][1]
    n = tn + fp + fn + tp
    rows.append(dict(arm=a, label=ARM_LABEL[a], n_epochs=int(n),
                     tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp),
                     sensitivity=tp / (tp + fn), specificity=tn / (tn + fp),
                     precision=tp / max(tp + fp, 1), npv=tn / max(tn + fn, 1),
                     fpr=fp / (fp + tn), fnr=fn / (fn + tp),
                     accuracy=(tp + tn) / n))
CM_DF = pd.DataFrame(rows)
CM_DF.to_csv(os.path.join(CMP_CSV, "confusion_matrices.csv"), index=False)
print(CM_DF.to_string(index=False, float_format=lambda v: f"{v:9.4f}"))

fig, axes = plt.subplots(2, len(ARMS_DONE), figsize=(4.1 * len(ARMS_DONE), 8))
axes = np.atleast_2d(axes)
for j, a in enumerate(ARMS_DONE):
    cm = CONFUSION[a].astype(float)
    for i, (mat, ttl, fmt, vmax) in enumerate((
            (cm, "counts", lambda v: f"{int(v):,}", cm.max()),
            (cm / cm.sum(1, keepdims=True), "row-normalized",
             lambda v: f"{v:.3f}", 1.0))):
        ax = axes[i, j]
        im = ax.imshow(mat, cmap="Blues", vmin=0, vmax=vmax)
        for r in range(2):
            for c in range(2):
                ax.text(c, r, fmt(mat[r, c]), ha="center", va="center", fontsize=11,
                        color="white" if mat[r, c] > 0.55 * vmax else "black")
        ax.set_xticks([0, 1]); ax.set_xticklabels(["pred 0", "pred 1"])
        ax.set_yticks([0, 1]); ax.set_yticklabels(["true 0", "true 1"])
        ax.set_title(f"{ARM_LABEL[a]}\n{ttl}", fontsize=10)
fig.suptitle("Pooled confusion matrices — all test epochs, each patient at its own threshold")
fig.tight_layout()
fig.savefig(os.path.join(CMP_FIG, "01_confusion_matrices.png"), dpi=150, bbox_inches="tight")
plt.show()

## Cell 12 — Paired significance tests

The reason this design is worth running: **every arm is evaluated on the same 89
patients**, so the results are paired and differences can be tested directly rather than
eyeballed.

### Wilcoxon signed-rank, not a t-test

Per-patient F1 is bounded in [0, 1], skewed, and has a hard floor at 0 that several
patients actually hit. A paired t-test assumes the *differences* are roughly normal,
which they are not here. The **Wilcoxon signed-rank test** ranks the absolute
differences and tests whether positives and negatives are balanced — no distributional
assumption, and it is the standard choice for paired model comparison on bounded metrics.

### Holm correction

13 metrics × 3 comparisons = 39 tests. At α = 0.05 you would expect ~2 false positives
by chance alone. **Holm–Bonferroni** sorts the p-values and tests the $i$-th against
$\alpha/(m - i + 1)$ — uniformly more powerful than plain Bonferroni while controlling
the same family-wise error rate. Both raw and adjusted p-values are reported; quote the
adjusted ones.

### Effect size, because p-values alone are not enough

`median_delta` is the median per-patient change against BCE, in the metric's own units —
the number a clinician cares about. Deltas for Brier and ECE are **sign-flipped** so
that a positive delta always reads as "this arm improves on BCE", regardless of whether
the metric's raw direction is higher-better or lower-better. `rank_biserial` is the standardized effect size for
Wilcoxon: the proportion of patients favouring the arm minus the proportion favouring
the baseline, so +0.4 means the arm wins on 70% of patients and loses on 30%. A
significant p with a tiny median delta means "reliably, but not meaningfully, better" —
and with n=89 that combination is common enough to be worth checking every time.

In [ ]:
# =========================================================================
# CELL 12 — paired Wilcoxon tests vs the BCE baseline
# =========================================================================
def holm(pvals):
    """Holm-Bonferroni: uniformly more powerful than Bonferroni, same FWER."""
    p = np.asarray(pvals, float)
    m = len(p)
    order = np.argsort(p)
    adj = np.empty(m)
    running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, (m - rank) * p[idx])
        adj[idx] = min(running, 1.0)
    return adj


TEST_METRICS = ["accuracy", "precision", "recall", "sensitivity", "specificity",
                "f1", "balanced_accuracy", "mcc", "cohen_kappa", "roc_auc",
                "pr_auc", "brier", "ece"]
LOWER_IS_BETTER = {"brier", "ece"}      # deltas for these are sign-flipped so that
                                        # positive delta ALWAYS means "arm improves"

base = ARM_DF.get(BASELINE_ARM)
rows = []
if base is not None:
    for arm in ARMS_DONE:
        if arm == BASELINE_ARM:
            continue
        merged = base[["patient"] + TEST_METRICS].merge(
            ARM_DF[arm][["patient"] + TEST_METRICS], on="patient",
            suffixes=("_base", "_arm"))
        for m in TEST_METRICS:
            b = merged[f"{m}_base"].astype(float).values
            v = merged[f"{m}_arm"].astype(float).values
            ok = np.isfinite(b) & np.isfinite(v)
            b, v = b[ok], v[ok]
            d = (b - v) if m in LOWER_IS_BETTER else (v - b)   # + = improvement
            nz = d[d != 0]
            if len(nz) < 5:
                stat, p = np.nan, np.nan
                rb = np.nan
            else:
                stat, p = wilcoxon(v, b, zero_method="wilcox")
                rb = (np.sign(nz) > 0).mean() - (np.sign(nz) < 0).mean()
            rows.append(dict(arm=arm, label=ARM_LABEL[arm], metric=m, n_paired=len(b),
                             baseline_mean=b.mean(), arm_mean=v.mean(),
                             mean_delta=d.mean(), median_delta=np.median(d),
                             n_improved=int((d > 0).sum()), n_degraded=int((d < 0).sum()),
                             rank_biserial=rb, p_raw=p))

TESTS = pd.DataFrame(rows)
if len(TESTS):
    TESTS["p_holm"] = holm(TESTS.p_raw.fillna(1.0).values)
    TESTS["sig"] = np.where(TESTS.p_holm < 0.001, "***",
                     np.where(TESTS.p_holm < 0.01, "**",
                       np.where(TESTS.p_holm < 0.05, "*", "ns")))
    TESTS.to_csv(os.path.join(CMP_CSV, "paired_tests_vs_bce.csv"), index=False)

    print("=" * 118)
    print(f"  PAIRED WILCOXON vs {ARM_LABEL[BASELINE_ARM]} "
          f"(n={TESTS.n_paired.max()} patients, Holm-corrected over {len(TESTS)} tests)")
    print("=" * 118)
    for arm in [a for a in ARMS_DONE if a != BASELINE_ARM]:
        t = TESTS[TESTS.arm == arm]
        print(f"\n--- {ARM_LABEL[arm]} ---")
        print(t[["metric", "baseline_mean", "arm_mean", "median_delta",
                 "n_improved", "n_degraded", "rank_biserial", "p_raw", "p_holm", "sig"]]
              .to_string(index=False, float_format=lambda v: f"{v:9.4f}"))
    n_sig = int((TESTS.p_holm < 0.05).sum())
    print(f"\n{n_sig} of {len(TESTS)} comparisons significant after Holm correction")
else:
    print(f"baseline arm '{BASELINE_ARM}' not available -- skipping paired tests")

## Cell 13 — Comparison figures

Nine figures in `comparison/figures/`, at 150 dpi.

1. **Grouped metric bars** with SD error bars — the headline picture. Overlapping error
   bars mean the difference is not real regardless of how the point estimates look.
2. **Per-patient distributions** — violin/box per arm for F1, AUROC, MCC and κ. The
   spread matters as much as the mean; an arm with a higher mean and a longer bottom tail
   is worse for screening.
3. **Paired slope plot** — one line per patient from BCE to each arm. Crossing lines mean
   the arms disagree per patient rather than one dominating; parallel lines mean a uniform
   shift. This is the picture the Wilcoxon test summarizes.
4. **Pooled ROC** overlay, all arms.
5. **Pooled PR** overlay, with the prevalence chance floor marked.
6. **Reliability diagrams** overlaid, with ECE in the legend. Expect the focal arms to bow
   away from the diagonal — down-weighting easy examples distorts the probability scale.
7. **Events per hour: true vs predicted**, one panel per arm, with the identity line, the
   fitted line, R² and MAE. The gap between the two lines *is* the bias.
8. **Bland–Altman**, one panel per arm, shared axes so the limits of agreement are
   directly comparable.
9. **Severity strata** — F1 and AUROC by band, grouped by arm. Watch whether AUROC stays
   flat across bands while F1 moves; that pattern is the operating-point signature and
   whether it survives across all four losses is the interesting question.

In [ ]:
# =========================================================================
# CELL 13 — comparison figures
# =========================================================================
def _save(fig, name):
    fig.tight_layout()
    fig.savefig(os.path.join(CMP_FIG, f"{name}.png"), dpi=150, bbox_inches="tight")
    plt.show()

L = [ARM_LABEL[a] for a in ARMS_DONE]
C = [ARM_COLOR[a] for a in ARMS_DONE]

# --- 2 grouped metric bars ----------------------------------------------
bars = ["accuracy", "precision", "sensitivity", "specificity", "f1",
        "balanced_accuracy", "mcc", "cohen_kappa", "roc_auc", "pr_auc"]
fig, ax = plt.subplots(figsize=(14, 5.5))
x = np.arange(len(bars)); w = 0.8 / len(ARMS_DONE)
for k, a in enumerate(ARMS_DONE):
    mu = [np.nanmean(ARM_DF[a][m]) for m in bars]
    sd = [np.nanstd(ARM_DF[a][m], ddof=1) for m in bars]
    ax.bar(x + k * w - 0.4 + w / 2, mu, w, yerr=sd, capsize=2.5,
           label=ARM_LABEL[a], color=ARM_COLOR[a], alpha=.9,
           error_kw=dict(lw=.8, alpha=.6))
ax.set_xticks(x); ax.set_xticklabels(bars, rotation=25, ha="right")
ax.set_ylabel("mean over patients (± SD)"); ax.set_ylim(0, 1.05)
ax.set_title("Loss ablation — LOPO means with between-patient SD")
ax.legend(ncol=len(ARMS_DONE)); ax.grid(axis="y", alpha=.3)
_save(fig, "02_metric_bars")

# --- 3 per-patient distributions ----------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(17, 4.8))
for ax, m in zip(axes, ["f1", "roc_auc", "mcc", "cohen_kappa"]):
    data = [ARM_DF[a][m].dropna().values for a in ARMS_DONE]
    bp = ax.boxplot(data, showmeans=True, patch_artist=True,
                    **({"tick_labels": L} if matplotlib.__version__ >= "3.9"
                       else {"labels": L}))
    for patch, col in zip(bp["boxes"], C):
        patch.set_facecolor(col); patch.set_alpha(.55)
    ax.set_title(m); ax.grid(axis="y", alpha=.3)
    ax.tick_params(axis="x", rotation=25)
fig.suptitle(f"Per-patient distribution across {len(ARM_DF[ARMS_DONE[0]])} folds")
_save(fig, "03_distributions")

# --- 4 paired slope plot -------------------------------------------------
if BASELINE_ARM in ARM_DF and len(ARMS_DONE) > 1:
    others = [a for a in ARMS_DONE if a != BASELINE_ARM]
    fig, axes = plt.subplots(1, len(others), figsize=(4.6 * len(others), 5), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, arm in zip(axes, others):
        mg = (ARM_DF[BASELINE_ARM][["patient", "f1"]]
              .merge(ARM_DF[arm][["patient", "f1"]], on="patient",
                     suffixes=("_b", "_a")).dropna())
        for _, r in mg.iterrows():
            ax.plot([0, 1], [r.f1_b, r.f1_a], lw=.6,
                    c=("#59a14f" if r.f1_a >= r.f1_b else "#d1495b"), alpha=.55)
        ax.plot([0, 1], [mg.f1_b.mean(), mg.f1_a.mean()], "o-", lw=2.6, c="k",
                label=f"mean {mg.f1_b.mean():.3f} -> {mg.f1_a.mean():.3f}")
        ax.set_xticks([0, 1]); ax.set_xticklabels([ARM_LABEL[BASELINE_ARM], ARM_LABEL[arm]])
        ax.set_title(f"{int((mg.f1_a > mg.f1_b).sum())} better / "
                     f"{int((mg.f1_a < mg.f1_b).sum())} worse")
        ax.legend(fontsize=8); ax.grid(axis="y", alpha=.3)
    axes[0].set_ylabel("per-patient F1")
    fig.suptitle("Paired per-patient F1 — green improves on BCE, red degrades")
    _save(fig, "04_paired_slopes")

# --- 5/6 pooled ROC and PR ----------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for a in ARMS_DONE:
    r = POOLED_RAW[a]
    fpr, tpr, _ = roc_curve(r["labels"], r["probs"])
    axes[0].plot(fpr, tpr, lw=1.7, c=ARM_COLOR[a],
                 label=f"{ARM_LABEL[a]} ({POOLED[a]['roc_auc']:.4f})")
    pr, rc, _ = precision_recall_curve(r["labels"], r["probs"])
    axes[1].plot(rc, pr, lw=1.7, c=ARM_COLOR[a],
                 label=f"{ARM_LABEL[a]} ({POOLED[a]['pr_auc']:.4f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=.8)
axes[0].set_xlabel("false positive rate"); axes[0].set_ylabel("true positive rate")
axes[0].set_title("Pooled ROC"); axes[0].legend(fontsize=8); axes[0].grid(alpha=.3)
prev = POOLED_RAW[ARMS_DONE[0]]["labels"].mean()
axes[1].axhline(prev, ls="--", c="k", lw=.8, label=f"chance = prevalence {prev:.3f}")
axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision")
axes[1].set_title("Pooled precision-recall"); axes[1].legend(fontsize=8); axes[1].grid(alpha=.3)
_save(fig, "05_roc_pr")

# --- 7 reliability -------------------------------------------------------
fig, ax = plt.subplots(figsize=(6.5, 6))
edges = np.linspace(0, 1, N_CALIB_BINS + 1)
for a in ARMS_DONE:
    r = POOLED_RAW[a]; xs, ys = [], []
    for i in range(N_CALIB_BINS):
        hi = r["probs"] <= edges[i+1] if i == N_CALIB_BINS-1 else r["probs"] < edges[i+1]
        m = (r["probs"] >= edges[i]) & hi
        if m.any():
            xs.append(r["probs"][m].mean()); ys.append(r["labels"][m].mean())
    ax.plot(xs, ys, "o-", lw=1.5, ms=4, c=ARM_COLOR[a],
            label=f"{ARM_LABEL[a]} (ECE {POOLED[a]['ece']:.4f})")
ax.plot([0, 1], [0, 1], "k--", lw=.8, label="perfect")
ax.set_xlabel("mean predicted probability"); ax.set_ylabel("observed frequency")
ax.set_title("Reliability — below the diagonal = overconfident")
ax.legend(fontsize=8); ax.grid(alpha=.3)
_save(fig, "06_calibration")

# --- 8 events per hour: true vs predicted --------------------------------
fig, axes = plt.subplots(1, len(ARMS_DONE), figsize=(4.4 * len(ARMS_DONE), 4.6),
                         sharex=True, sharey=True)
axes = np.atleast_1d(axes)
lim = max(ALL.true_eph.max(), ALL.pred_eph.max()) * 1.08
for ax, a in zip(axes, ARMS_DONE):
    d = ARM_DF[a]
    ax.scatter(d.true_eph, d.pred_eph, s=32, c=ARM_COLOR[a], alpha=.8)
    ax.plot([0, lim], [0, lim], "k--", lw=1, label="identity")
    z = np.polyfit(d.true_eph, d.pred_eph, 1)
    ax.plot([0, lim], np.polyval(z, [0, lim]), "-", c="grey", lw=1.3, label="fit")
    for b in EPH_BINS:
        ax.axvline(b, c="#bbb", lw=.6); ax.axhline(b, c="#bbb", lw=.6)
    ax.set_title(f"{ARM_LABEL[a]}\nR2={EPH[a]['eph_r2_determination']:.3f}  "
                 f"r={EPH[a]['eph_pearson_r']:.3f}  MAE={EPH[a]['eph_mae']:.2f}/h",
                 fontsize=10)
    ax.set_xlabel("true events/h"); ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.legend(fontsize=7); ax.grid(alpha=.3)
axes[0].set_ylabel("predicted events/h")
fig.suptitle("Events per hour — distance from the dashed line is the error")
_save(fig, "07_events_per_hour")

# --- 9 Bland-Altman ------------------------------------------------------
fig, axes = plt.subplots(1, len(ARMS_DONE), figsize=(4.4 * len(ARMS_DONE), 4.6),
                         sharey=True)
axes = np.atleast_1d(axes)
for ax, a in zip(axes, ARMS_DONE):
    d = ARM_DF[a]
    diff = d.pred_eph - d.true_eph
    mn = (d.pred_eph + d.true_eph) / 2
    bias, sd = diff.mean(), diff.std(ddof=1)
    ax.scatter(mn, diff, s=32, c=ARM_COLOR[a], alpha=.8)
    ax.axhline(bias, c="k", lw=1.2)
    ax.axhline(bias + 1.96 * sd, c="r", ls="--", lw=1)
    ax.axhline(bias - 1.96 * sd, c="r", ls="--", lw=1)
    ax.axhline(0, c="grey", lw=.7)
    ax.set_title(f"{ARM_LABEL[a]}\nbias {bias:+.2f}  LoA [{bias-1.96*sd:+.2f}, "
                 f"{bias+1.96*sd:+.2f}]", fontsize=10)
    ax.set_xlabel("mean of true and predicted (events/h)"); ax.grid(alpha=.3)
axes[0].set_ylabel("predicted − true")
fig.suptitle("Bland–Altman, shared y-axis")
_save(fig, "08_bland_altman")

# --- 10 severity strata --------------------------------------------------
sev_present = [s for s in SEVERITY_NAMES if s in set(STRATA.severity)]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, metric in zip(axes, ["f1", "roc_auc"]):
    x = np.arange(len(sev_present)); w = 0.8 / len(ARMS_DONE)
    for k, a in enumerate(ARMS_DONE):
        vals = [STRATA[(STRATA.arm == a) & (STRATA.severity == s)][metric].mean()
                for s in sev_present]
        ax.bar(x + k * w - 0.4 + w / 2, vals, w, label=ARM_LABEL[a],
               color=ARM_COLOR[a], alpha=.9)
    ax.set_xticks(x); ax.set_xticklabels(sev_present)
    ax.set_ylim(0, 1.05); ax.set_ylabel(metric); ax.set_title(f"{metric} by severity")
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=.3)
fig.suptitle("Flat AUROC with moving F1 across bands = operating-point effect")
_save(fig, "09_severity_strata")

print(f"figures -> {CMP_FIG}")

## Cell 14 — Report

The paper table, the factorial read-out, and an inventory of what was written.

### Reading the 2 × 2

Because the design is factorial, two contrasts are computed directly:

- **Effect of weighting** = mean(weighted arms) − mean(unweighted arms)
- **Effect of focal** = mean(focal arms) − mean(non-focal arms)

These are more useful than "which arm won", because they tell you *why*. An interaction —
weighting helping under BCE but hurting under focal — shows up as the two arms disagreeing
about the sign, and is worth a sentence in the paper if present.

### What to report

Lead with **macro mean ± SD over 89 patients**, with the Holm-adjusted p-value against
BCE. Give ROC-AUC and PR-AUC as the discrimination pair, F1/MCC at the operating point,
ECE for calibration, and events-per-hour **R²** (determination, not squared correlation)
plus MAE and severity κ_w as the clinical endpoints. If AUROC barely moves across arms
while F1, ECE and κ_w move a lot, that is the finding — the loss is changing *where the
probabilities sit*, not how well the model ranks epochs, and that is the operating-point
argument made from a second direction.

In [ ]:
# =========================================================================
# CELL 14 — report
# =========================================================================
print("=" * 100)
print(f"  LOSS ABLATION REPORT — {len(ARMS_DONE)} arms, "
      f"{len(ARM_DF[ARMS_DONE[0]])} LOPO folds each")
print("=" * 100)

PAPER = ["f1", "sensitivity", "specificity", "precision", "mcc",
         "cohen_kappa", "roc_auc", "pr_auc", "ece"]
for m in PAPER:
    line = f"  {m:20s}"
    for a in ARMS_DONE:
        v = ARM_DF[a][m].astype(float)
        star = ""
        if len(TESTS):
            t = TESTS[(TESTS.arm == a) & (TESTS.metric == m)]
            if len(t):
                star = t.iloc[0]["sig"].replace("ns", "")
        line += f" | {ARM_LABEL[a][:6]:>6s} {np.nanmean(v):.4f}±{np.nanstd(v, ddof=1):.3f}{star:<3s}"
    print(line)

print("\n  events per hour")
for k, lab in [("eph_pearson_r", "Pearson r"), ("eph_r2_determination", "R2"),
               ("eph_mae", "MAE /h"), ("eph_bias", "bias /h"),
               ("severity_kappa_w", "severity kappa_w")]:
    print(f"  {lab:20s}" + "".join(f" | {ARM_LABEL[a][:6]:>6s} {EPH[a][k]:+.4f}"
                                   for a in ARMS_DONE))

# ---- 2x2 factorial contrasts -------------------------------------------
def _mean(arms, metric):
    vals = [np.nanmean(ARM_DF[a][metric]) for a in arms if a in ARM_DF]
    return float(np.mean(vals)) if vals else np.nan

W_ON = [a for a in ARMS_DONE if a.startswith("weighted")]
W_OFF = [a for a in ARMS_DONE if not a.startswith("weighted")]
F_ON = [a for a in ARMS_DONE if "focal" in a]
F_OFF = [a for a in ARMS_DONE if "focal" not in a]

if W_ON and W_OFF and F_ON and F_OFF:
    rows = []
    for m in PAPER:
        sgn = -1.0 if m in LOWER_IS_BETTER else 1.0     # + = mechanism helps, always
        rows.append(dict(metric=m,
                         effect_of_weighting=sgn * (_mean(W_ON, m) - _mean(W_OFF, m)),
                         effect_of_focal=sgn * (_mean(F_ON, m) - _mean(F_OFF, m))))
    FACT = pd.DataFrame(rows)
    FACT.to_csv(os.path.join(CMP_CSV, "factorial_effects.csv"), index=False)
    print("\n---- 2x2 FACTORIAL EFFECTS (positive = the mechanism helps; "
      "Brier/ECE sign-flipped accordingly) ----")
    print(FACT.to_string(index=False, float_format=lambda v: f"{v:+9.4f}"))

# ---- discrimination vs operating point ----------------------------------
auc_span = max(np.nanmean(ARM_DF[a].roc_auc) for a in ARMS_DONE) - \
           min(np.nanmean(ARM_DF[a].roc_auc) for a in ARMS_DONE)
f1_span = max(np.nanmean(ARM_DF[a].f1) for a in ARMS_DONE) - \
          min(np.nanmean(ARM_DF[a].f1) for a in ARMS_DONE)
ece_span = max(np.nanmean(ARM_DF[a].ece) for a in ARMS_DONE) - \
           min(np.nanmean(ARM_DF[a].ece) for a in ARMS_DONE)
kap = [EPH[a]["severity_kappa_w"] for a in ARMS_DONE if np.isfinite(EPH[a]["severity_kappa_w"])]
print(f"\n---- SPAN ACROSS THE FOUR LOSSES ----")
print(f"  AUROC            {auc_span:.4f}   <- discrimination")
print(f"  F1               {f1_span:.4f}   <- at the operating point")
print(f"  ECE              {ece_span:.4f}   <- calibration")
if kap:
    print(f"  severity kappa_w {max(kap)-min(kap):.4f}   <- clinical endpoint")
print("  A small AUROC span beside large F1/ECE/kappa spans means the loss moves WHERE")
print("  the probabilities sit, not how well the model ranks epochs.")

# ---- inventory ----------------------------------------------------------
print(f"\n  output root: {OUT_DIR}")
for a in ARMS_DONE:
    nf = len(glob.glob(os.path.join(arm_dir(a, "folds"), "*.npz")))
    print(f"    {a:16s} {nf:3d} folds | per_patient.csv | summary.csv")
print(f"    comparison/csv     {len(glob.glob(os.path.join(CMP_CSV,'*.csv'))):3d} files")
print(f"    comparison/figures {len(glob.glob(os.path.join(CMP_FIG,'*.png'))):3d} PNGs")
for f in sorted(glob.glob(os.path.join(CMP_CSV, "*.csv"))):
    print(f"      {os.path.basename(f)}")